# 03 - Modelleme

Bu bölümde Logistic Regression, Random Forest ve XGBoost modellerini aynı veri protokolünde karşılaştırıyoruz.

Karşılaştırma metrikleri:
- Accuracy
- Precision
- Recall
- F1
- ROC AUC
- Brier


In [ ]:
# Ortamı hazırlıyor ve proje kökünü Python yoluna ekliyoruz.
from pathlib import Path
import sys

proje_kok = Path.cwd().resolve()
while proje_kok.name != 'diyabet_risk_tahmini' and proje_kok.parent != proje_kok:
    proje_kok = proje_kok.parent

if proje_kok.name != 'diyabet_risk_tahmini':
    raise RuntimeError('Notebook proje kökünden veya alt klasörlerinden çalıştırılmalıdır.')

if str(proje_kok) not in sys.path:
    sys.path.insert(0, str(proje_kok))

print(f'Proje kökü: {proje_kok}')


In [ ]:
# Modelleme ve değerlendirme için gereken araçları içe aktarıyoruz.
import json
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_curve, auc, ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split

from makine_ogrenmesi.kaynak.model_egitimi import grid_searchleri_olustur
from makine_ogrenmesi.kaynak.model_degerlendirme import model_sonuc_ozeti_olustur
from makine_ogrenmesi.kaynak.ozellik_yapilandirmasi import HEDEF_KOLONU, OZELLIK_KOLONLARI
from makine_ogrenmesi.kaynak.veri_yukleyici import veri_setini_yukle


In [ ]:
# Veriyi eğitim ve test olarak ayırıp modelleme ortamını hazırlıyoruz.
veri = veri_setini_yukle(proje_kok / 'makine_ogrenmesi' / 'veri' / 'ham' / 'diabetes.csv')
X = veri[OZELLIK_KOLONLARI]
y = veri[HEDEF_KOLONU]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
print('Eğitim:', X_train.shape, 'Test:', X_test.shape)


In [ ]:
# Üç modeli GridSearch ile eğitip test metriklerini tek bir tabloda topluyoruz.
grid_searchler = grid_searchleri_olustur(scoring='roc_auc', n_jobs=-1)

sonuclar = []
egitilmis_modeller = {}

for model_adi, grid in grid_searchler.items():
    print(f'[{model_adi}] eğitiliyor...')
    grid.fit(X_train, y_train)
    model = grid.best_estimator_
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    ozet = model_sonuc_ozeti_olustur(model_adi, y_test, y_pred, y_prob)
    ozet['en_iyi_cv_roc_auc'] = float(grid.best_score_)
    ozet['en_iyi_parametreler'] = grid.best_params_

    sonuclar.append(ozet)
    egitilmis_modeller[model_adi] = model

sonuc_df = pd.DataFrame(sonuclar).sort_values(
    ['roc_auc', 'recall', 'f1', 'accuracy'],
    ascending=False,
).reset_index(drop=True)

sonuc_df


In [ ]:
# Sıralamaya göre öne çıkan modeli ve en iyi parametrelerini yazdırıyoruz.
en_iyi = sonuc_df.iloc[0].to_dict()
print('Seçilen model:', en_iyi['model_adi'])
print('En iyi CV ROC AUC:', round(float(en_iyi['en_iyi_cv_roc_auc']), 4))
print('Parametreler:', en_iyi['en_iyi_parametreler'])


In [ ]:
# Modelleri ROC eğrisi üzerinde yan yana görselleştirerek ayrıştırma performansını karşılaştırıyoruz.
fig, ax = plt.subplots(figsize=(7, 5))
for satir in sonuclar:
    model_adi = satir['model_adi']
    model = egitilmis_modeller[model_adi]
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{model_adi} (AUC={roc_auc:.3f})")

ax.plot([0, 1], [0, 1], '--', color='gray')
ax.set_title('Model ROC Karşılaştırması')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
ax.grid(alpha=0.3)
plt.show()


In [ ]:
# En iyi model için confusion matrix çizip hata profilini inceliyoruz.
en_iyi_model_adi = sonuc_df.loc[0, 'model_adi']
en_iyi_model = egitilmis_modeller[en_iyi_model_adi]
y_pred_best = en_iyi_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best, labels=[0, 1])

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap='Blues')
plt.title(f'Confusion Matrix - {en_iyi_model_adi}')
plt.show()


In [ ]:
# Notebook içinde üretilen model özetini ayrı bir JSON dosyasına kaydediyoruz.
cikti_yolu = proje_kok / 'makine_ogrenmesi' / 'raporlar' / 'degerlendirme' / 'modelleme_notebook_ozeti.json'
cikti_yolu.parent.mkdir(parents=True, exist_ok=True)

payload = {
    'sirali_sonuclar': sonuc_df.to_dict(orient='records'),
    'en_iyi_model': sonuc_df.iloc[0].to_dict(),
}

cikti_yolu.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('Özet yazıldı:', cikti_yolu)


## Kısa değerlendirme

- Buradaki karşılaştırma, model seçimini tek bir metriğe hapsetmeden yapar.
- Brier metriğini görmemiz, olasılık kalitesini ayrıca yorumlamamızı sağlar.
- Sonraki adımda kalibrasyon ve eşik analizi ile deploy davranışını netleştireceğiz.
